In [0]:
-- 1. Ingest Airbnb CSV into Bronze Delta Table all as string 
CREATE OR REPLACE TABLE bronze.airbnb_raw
USING DELTA
AS
SELECT
  *,
  current_timestamp() AS _ingested_at,
  _metadata.file_path AS _source_file
FROM read_files(
  '/Volumes/workspace/bronze/bronze_feeding/airbnb.csv',
  format => 'csv',
  header => true,
  inferSchema => false
);

In [0]:
-- 2. Ingest Kamernet JSON (multiLine array) into Bronze Delta Table
CREATE OR REPLACE TABLE bronze.rentals_raw
USING DELTA
AS
SELECT
  *,
  current_timestamp() AS _ingested_at,
  _metadata.file_path AS _source_file
FROM read_files(
  '/Volumes/workspace/bronze/bronze_feeding/rentals.json',
  format => 'json',
  multiLine => true
);

In [0]:
-- Create Bronze Delta Table using Databricks SQL
CREATE OR REPLACE TABLE bronze.post_codes_raw AS
SELECT
  feature.properties.pc4_code::STRING AS pc4_code,
  feature.properties.year::INT AS year,
  feature.properties.gem_code::STRING AS gem_code,
  feature.properties.gem_name::STRING AS gem_name,
  feature.properties.prov_code::STRING AS prov_code,
  feature.properties.prov_name::STRING AS prov_name,
  feature.properties.geo_point_2d.lat::DOUBLE AS centroid_lat,
  feature.properties.geo_point_2d.lon::DOUBLE AS centroid_lon,
  to_json(feature.geometry) AS raw_geometry_json,
  current_timestamp() AS _ingested_at
FROM read_files(
  '/Volumes/workspace/bronze/bronze_feeding/post_codes.geojson',
  format => 'json',
  multiLine => true
)
LATERAL VIEW explode(features) AS feature;